In [1]:
import os
from pathlib import Path
import re

import duckdb
from dotenv import load_dotenv
import pandas as pd

In [11]:
def infer_table_order(schema_file_path):
    """Infer table order based on foreign key dependencies."""
    # Read the schema from the file
    with open(schema_file_path, 'r') as file:
        schema_sql = file.read()

    tables = {}
    dependencies = {}
    
    # Remove comments
    schema_sql = re.sub(r'--.*$', '', schema_sql, flags=re.MULTILINE)
    
    # Split the schema into individual CREATE TABLE statements
    create_statements = re.split(r'(?i)CREATE\s+OR\s+REPLACE\s+TABLE', schema_sql)
    #print("\nCreate statements:")
    
    for statement in create_statements[1:]:  # Skip the first empty split
        # print(statement.strip())
        # Extract table name
        match = re.match(r'\s*(\w+)', statement)
        if match:
            current_table = match.group(1)
            tables[current_table] = set()
            dependencies[current_table] = set()
            
            # Find all FOREIGN KEY references
            fk_references = re.findall(r'FOREIGN KEY.*REFERENCES\s+(\w+)', statement, re.IGNORECASE)
            for referenced_table in fk_references:
                tables[current_table].add(referenced_table)
                dependencies[current_table].add(referenced_table)
    
    # Topological sort
    def topological_sort(graph):
        visited = set()
        result = []
        
        def dfs(node):
            if node in visited:
                return
            visited.add(node)
            for neighbor in graph.get(node, []):
                if neighbor not in visited:
                    dfs(neighbor)
            result.append(node)
        
        for node in graph:
            if node not in visited:
                dfs(node)
        
        return result[::-1]  # Reverse to get correct order
    
    return topological_sort(dependencies)


In [12]:
SCHEMA = Path("../sql/create_cb_experiment_tables_ddb.sql")

In [13]:
table_order = infer_table_order(str(SCHEMA))
print("\nInferred table order:", table_order)


Create statements:
ExperimentCBs (
    ExperimentID INTEGER PRIMARY KEY,
    Cohort VARCHAR,
    SubjectName VARCHAR,
    SubjectNumber INTEGER,
    SessionNumber INTEGER,
    SessionType VARCHAR,  
    ExperimentType VARCHAR,
    Direction VARCHAR,
    Comment VARCHAR,
    DateTime TIMESTAMP,
    LogFileName VARCHAR,
    Parameters JSON
);
TrialCBs (
    TrialID INTEGER PRIMARY KEY,
    ExperimentID INTEGER,
    TrialNumber INTEGER,
    ResponseType VARCHAR,
    TrialSubType VARCHAR,
    CriterionType VARCHAR,
    CriterionCount DOUBLE,
    TrialStartTime TIMESTAMP,
    TrialEndTime TIMESTAMP,
    FOREIGN KEY (ExperimentID) REFERENCES ExperimentCBs (ExperimentID)
);
ResponseCBs (
    ResponseID INTEGER PRIMARY KEY,
    TrialID INTEGER,
    ResponseTime DOUBLE,
    EventDateTime TIMESTAMP,
    FOREIGN KEY (TrialID) REFERENCES TrialCBs (TrialID)
);
EventCBs (
    EventID INTEGER PRIMARY KEY,
    TrialID INTEGER,
    EventType VARCHAR,
    EventTime TIMESTAMP,
    ElapsedTime DOUBLE,
  

In [5]:
def copy_database_to_motherduck(source_db_path, md_db_name, tables=None):
    conn = duckdb.connect(':memory:')
    load_dotenv()

    md_token = os.getenv('MD_TOKEN')
    os.environ["motherduck_token"] = md_token
    if not md_token:
        raise ValueError("MotherDuck token not found in .env file")
        
    conn.sql("ATTACH 'md:';")

    try:
        # Attach the source database
        print(f"ATTACH '{source_db_path}' AS local_source_db")
        conn.execute(f"ATTACH '{source_db_path}' AS local_source_db")

        print("USE 'local_source_db'")
        conn.execute("USE 'local_source_db'")

            # Get the list of tables from the source database
        if tables is None:
            tables = [table[0] for table in conn.execute("SHOW TABLES").fetchall()]
        
        # Create the new database
        print(f"CREATE OR REPLACE DATABASE {md_db_name}")
        conn.execute(f"CREATE OR REPLACE DATABASE {md_db_name}")
        
        # Copy the schema from the source database to the new database
        print(f"COPY FROM DATABASE local_source_db TO {md_db_name} (SCHEMA)")
        conn.execute(f"COPY FROM DATABASE local_source_db TO {md_db_name} (SCHEMA)")
        
        # Copy data for each table
        print(tables)
        for table_name in tables:
            print(f"INSERT INTO {md_db_name}.{table_name} SELECT * FROM local_source_db.{table_name}")
            conn.execute(f"INSERT INTO {md_db_name}.{table_name} SELECT * FROM local_source_db.{table_name}")
        
        print(f"INFO: Database '{md_db_name}' created and populated successfully.")
    
    except Exception as e:
        print(f"ERROR: An error occurred: {str(e)}")
    
    finally:
        conn.close()

    return md_db_name

In [3]:
# Order of inserting data into tables - if not this then the insert fails

table_order = ['ExperimentCBs', 'TrialCBs', 'ResponseCBs', 'EventCBs']

In [7]:
CB_DATABASE = Path("../data/Experiments_CB_2023_Q4.ddb")

In [8]:
CB_DATABASE_MD = str(CB_DATABASE.stem)

In [ ]:
md_db = copy_database_to_motherduck(CB_DATABASE, CB_DATABASE_MD, table_order)

In [13]:
def compare_databases(original_db, md_db):
    print(f"Comparing - ORIGINAL: {original_db} vs NEW: {md_db}")
    conn_original = duckdb.connect(original_db)
    conn_new = duckdb.connect(md_db)

    tables_original = conn_original.execute("SHOW TABLES").fetchall()
    tables_new = conn_new.execute("SHOW TABLES").fetchall()

    if len(tables_original) != len(tables_new):
        print(f"Number of tables mismatch. Original: {len(tables_original)}, New: {len(tables_new)}")
        return False

    for table in tables_original:
        table_name = table[0]
        print(f"Comparing table: {table_name}")

        schema_original = conn_original.execute(f"DESCRIBE {table_name}").fetchall()
        schema_new = conn_new.execute(f"DESCRIBE {table_name}").fetchall()

        if schema_original != schema_new:
            print(f"Schema mismatch for table {table_name}")
            print("Original schema:")
            print(pd.DataFrame(schema_original, columns=['Column', 'Type', 'Null', 'Key', 'Default', 'Extra']))
            print("\nNew schema:")
            print(pd.DataFrame(schema_new, columns=['Column', 'Type', 'Null', 'Key', 'Default', 'Extra']))
            return False

        # Compare data
        data_original = conn_original.execute(f"SELECT * FROM {table_name} ORDER BY 1").fetchall()
        data_new = conn_new.execute(f"SELECT * FROM {table_name} ORDER BY 1").fetchall()

        if data_original != data_new:
            print(f"Data mismatch for table {table_name}")
            
            # For TrialCBs table, perform a more detailed comparison
            if table_name == 'TrialCBs':
                df_original = pd.DataFrame(data_original, columns=[col[0] for col in schema_original])
                df_new = pd.DataFrame(data_new, columns=[col[0] for col in schema_new])
                
                # Compare row counts
                if len(df_original) != len(df_new):
                    print(f"Row count mismatch. Original: {len(df_original)}, New: {len(df_new)}")
                
                # Compare column values
                for column in df_original.columns:
                    if not df_original[column].equals(df_new[column]):
                        print(f"Mismatch in column: {column}")
                        print("First few mismatching rows:")
                        mismatch = (df_original[column] != df_new[column])
                        print(pd.concat([df_original[mismatch], df_new[mismatch]], axis=1, keys=['Original', 'New']).head())
                
            return False

        # Compare row count
        count_original = conn_original.execute(f"SELECT COUNT(*) FROM {table_name}").fetchone()[0]
        count_new = conn_new.execute(f"SELECT COUNT(*) FROM {table_name}").fetchone()[0]

        if count_original != count_new:
            print(f"Row count mismatch for table {table_name}. Original: {count_original}, New: {count_new}")
            return False

    print("All tables and data match!")
    return True

In [ ]:
databases_identical = compare_databases(CB_DATABASE, md_db)
print(f"Databases are identical: {databases_identical}")